# FaceForensics++ C23

Manifest from the official source-video split, face extraction, backbone fine-tuning, feature extraction and the SVM stage (Sections 3.2 to 3.4, Tables 1 to 5).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import json, urllib.request, pandas as pd
from pathlib import Path

BASE = Path('/content/drive/MyDrive/deepfake_finetuning')
OUT  = BASE/'rebuild_2026'/'manifests'; OUT.mkdir(parents=True, exist_ok=True)
FFPP = BASE/'FaceForensics++_C23'
VIDEO_EXT = {'.mp4','.avi','.mov','.mkv'}

# --- see what's actually in the FF++ folder
files = [p for p in FFPP.rglob('*') if p.suffix.lower() in VIDEO_EXT]
print(len(files), 'videos')
from collections import Counter
for d, n in Counter(str(p.parent) for p in files).most_common(30):
    print(f'{n:6d}  {d}')

Mounted at /content/drive
6433 videos
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/original
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/Face2Face
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/Deepfakes
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/FaceSwap
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/NeuralTextures
  1000  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/FaceShifter
   433  /content/drive/MyDrive/deepfake_finetuning/FaceForensics++_C23/DeepFakeDetection


In [ ]:
import json, urllib.request, pandas as pd
from pathlib import Path

BASE = Path('/content/drive/MyDrive/deepfake_finetuning')
OUT  = BASE/'rebuild_2026'/'manifests'; OUT.mkdir(parents=True, exist_ok=True)
FFPP = BASE/'FaceForensics++_C23'
URL  = 'https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits/{}.json'

# official source-video split: 360/70/70 pairs = 720/140/140 videos
splits = {}
for part in ('train','val','test'):
    f = OUT/f'ffpp_{part}.json'
    if not f.exists(): urllib.request.urlretrieve(URL.format(part), f)
    splits[part] = json.load(open(f))
print({k: len(v) for k, v in splits.items()})

part_of, group_of = {}, {}
for part, pairs in splits.items():
    for a, b in pairs:
        pair = '_'.join(sorted((a, b)))          # keeps a video and its fakes together
        for vid in (a, b, f'{a}_{b}', f'{b}_{a}'):
            part_of[vid], group_of[vid] = part, pair

rows, dfd, unmatched = [], [], []
for p in FFPP.rglob('*'):
    if p.suffix.lower() not in {'.mp4','.avi','.mov','.mkv'}: continue
    method = p.parent.name
    if method == 'DeepFakeDetection':
        actors = '_'.join(p.stem.split('_')[:2])          # actor pair = group
        dfd.append(dict(video_path=str(p), video_id=p.stem, group_id=actors,
                        label=1, method='DeepFakeDetection', partition='dfd_holdout'))
        continue
    if p.stem not in part_of:
        unmatched.append(str(p)); continue
    rows.append(dict(video_path=str(p), video_id=p.stem, group_id=group_of[p.stem],
                     label=0 if method == 'original' else 1,
                     method='real' if method == 'original' else method,
                     partition=part_of[p.stem]))

df = pd.DataFrame(rows + dfd)
print(f'\nmatched {len(rows)}, DFD held out {len(dfd)}, unmatched {len(unmatched)}')
if unmatched: print('examples:', unmatched[:3])

main = df[df.partition != 'dfd_holdout']
for a in ('train','val','test'):
    for b in ('train','val','test'):
        if a < b:
            assert not (set(main[main.partition==a].group_id) &
                        set(main[main.partition==b].group_id)), f'{a}/{b} overlap'
print('\nno group overlap between partitions')
print(main.pivot_table(index='partition', columns='method',
                       values='video_id', aggfunc='count', fill_value=0))
print('\nvideos per partition:', main.groupby('partition').size().to_dict())
df.to_csv(OUT/'ffpp_manifest.csv', index=False)
print('wrote', OUT/'ffpp_manifest.csv')

{'train': 360, 'val': 70, 'test': 70}

matched 6000, DFD held out 433, unmatched 0

no group overlap between partitions
method     Deepfakes  Face2Face  FaceShifter  FaceSwap  NeuralTextures  real
partition                                                                   
test             140        140          140       140             140   140
train            720        720          720       720             720   720
val              140        140          140       140             140   140

videos per partition: {'test': 840, 'train': 4320, 'val': 840}
wrote /content/drive/MyDrive/deepfake_finetuning/rebuild_2026/manifests/ffpp_manifest.csv


In [ ]:
!pip -q install --no-deps facenet-pytorch==2.6.0
import torch, numpy as np
from facenet_pytorch import MTCNN
det = MTCNN(image_size=300, margin=20, keep_all=False, post_process=False,
            select_largest=True, device='cuda' if torch.cuda.is_available() else 'cpu')
print('MTCNN ready |', torch.__version__, np.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 33.1 MB/s eta 0:00:00
MTCNN ready | 2.11.0+cu128 2.1.3


Face extraction, first pass. The real frames from this pass are used. Fake frame names collided across manipulation methods here, which is why the detection rate reads above 100 %; the fake frames are re-extracted below with method-qualified names.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import cv2, numpy as np, pandas as pd, torch, time
from pathlib import Path
from PIL import Image
from facenet_pytorch import MTCNN

BASE   = Path('/content/drive/MyDrive/deepfake_finetuning')
OUT    = BASE/'rebuild_2026'
FRAMES = OUT/'frames'/'ffpp'; FRAMES.mkdir(parents=True, exist_ok=True)
(OUT/'frames').mkdir(parents=True, exist_ok=True)
K, SIZE, LIMIT = 20, 300, None

df = pd.read_csv(OUT/'manifests'/'ffpp_manifest.csv')
df = df[df.partition != 'dfd_holdout'].reset_index(drop=True)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
det = MTCNN(image_size=SIZE, margin=20, keep_all=False, post_process=False,
            select_largest=True, device=dev)
print('device', dev, '| videos', len(df), flush=True)

def faces(video, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(video).stem
    cap = cv2.VideoCapture(str(video))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want = set(np.linspace(0, max(n-1,0), min(n,K)).round().astype(int).tolist()) if n else set()
    written, i = [], 0
    while want and i <= max(want):
        ok, fr = cap.read()
        if not ok: break
        if i in want:
            img = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            try: face = det(img)
            except Exception: face = None
            if face is not None:
                a = face.permute(1,2,0).numpy()
                a = ((a - a.min())/(np.ptp(a)+1e-8)*255).astype('uint8')
                d = out_dir/f'{stem}__f{i:05d}.jpg'
                Image.fromarray(a).resize((SIZE,SIZE)).save(d, quality=95)
                written.append(str(d))
        i += 1
    cap.release()
    return written

todo = df if LIMIT is None else df.head(LIMIT)
rows, failed, t0 = [], [], time.time()
for n, r in enumerate(todo.itertuples(), 1):
    d = FRAMES/r.partition/('fake' if r.label else 'real')
    done = sorted(d.glob(f'{r.video_id}__f*.jpg'))
    try:
        paths = [str(p) for p in done] if done else faces(r.video_path, d)
    except Exception as e:
        failed.append((r.video_id, str(e))); paths = []
    rows += [dict(frame_path=p, video_id=r.video_id, group_id=r.group_id,
                  label=r.label, method=r.method, partition=r.partition) for p in paths]
    if n % 100 == 0:
        pd.DataFrame(rows).to_csv(OUT/'frames'/'ffpp_frames.csv', index=False)
        el = time.time()-t0
        print(f'{n}/{len(todo)} videos | {len(rows)} frames | '
              f'{el/60:.0f} min elapsed | ~{(el/n)*(len(todo)-n)/3600:.1f} h left',
              flush=True)

idx = pd.DataFrame(rows)
idx.to_csv(OUT/'frames'/'ffpp_frames.csv', index=False)
if failed:
    pd.DataFrame(failed, columns=['video_id','error']).to_csv(
        OUT/'frames'/'ffpp_failures.csv', index=False)

print(f'\nDONE: {len(idx)} frames from {idx.video_id.nunique()} videos, {len(failed)} failures')
print(idx.pivot_table(index='partition', columns='label',
                      values='frame_path', aggfunc='count', fill_value=0))
print(idx.groupby(['partition','method']).size())
print('detection rate %.1f%%' % (100*len(idx)/max(1, K*idx.video_id.nunique())))
for a in ('train','val','test'):
    for b in ('train','val','test'):
        if a < b:
            assert not (set(idx[idx.partition==a].group_id) &
                        set(idx[idx.partition==b].group_id)), f'{a}/{b} LEAK'
print('verified: no source-video group appears in two partitions')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
device cuda | videos 6000
100/6000 videos | 1999 frames | 7 min elapsed | ~7.1 h left
200/6000 videos | 3999 frames | 18 min elapsed | ~8.8 h left
300/6000 videos | 5999 frames | 29 min elapsed | ~9.0 h left
400/6000 videos | 7999 frames | 40 min elapsed | ~9.3 h left
500/6000 videos | 9999 frames | 52 min elapsed | ~9.6 h left
600/6000 videos | 11999 frames | 66 min elapsed | ~9.8 h left
700/6000 videos | 13999 frames | 78 min elapsed | ~9.9 h left
800/6000 videos | 15999 frames | 92 min elapsed | ~10.0 h left
900/6000 videos | 17998 frames | 106 min elapsed | ~10.0 h left
1000/6000 videos | 19998 frames | 118 min elapsed | ~9.8 h left
1100/6000 videos | 21998 frames | 127 min elapsed | ~9.4 h left
1200/6000 videos | 23998 frames | 136 min elapsed | ~9.1 h left
1300/6000 videos | 25998 frames | 147 min elapsed | ~8.9 h left
1400/6000 videos | 27998 frames | 

Fake frames re-extracted with method-qualified names.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import cv2, numpy as np, pandas as pd, torch, time
from pathlib import Path
from PIL import Image
from facenet_pytorch import MTCNN

BASE, K, SIZE = Path('/content/drive/MyDrive/deepfake_finetuning'), 20, 300
OUT    = BASE/'rebuild_2026'
DRIVEF = OUT/'frames'/'ffpp'
LOCAL  = Path('/content/ffpp_frames'); LOCAL.mkdir(parents=True, exist_ok=True)

# one scan instead of 5,000 globs
done = set()
for base in (DRIVEF, LOCAL):
    for part in ('train','val','test'):
        d = base/part/'fake'
        if d.exists():
            done |= {p.name.split('__f')[0] for p in d.iterdir()}
print(f'{len(done)} fake videos already extracted')

df = pd.read_csv(OUT/'manifests'/'ffpp_manifest.csv')
df = df[(df.partition != 'dfd_holdout') & (df.label == 1)].reset_index(drop=True)
df['uid'] = df.method + '__' + df.video_id
todo = df[~df.uid.isin(done)].reset_index(drop=True)
print(f'{len(todo)} remaining')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
det = MTCNN(image_size=SIZE, margin=20, keep_all=False, post_process=False,
            select_largest=True, device=dev)

def faces(video, out_dir, uid):
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    want = set(np.linspace(0, max(n-1,0), min(n,K)).round().astype(int).tolist()) if n else set()
    written, i = [], 0
    while want and i <= max(want):
        ok, fr = cap.read()
        if not ok: break
        if i in want:
            img = Image.fromarray(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB))
            try: face = det(img)
            except Exception: face = None
            if face is not None:
                a = face.permute(1,2,0).numpy()
                a = ((a-a.min())/(np.ptp(a)+1e-8)*255).astype('uint8')
                d = out_dir/f'{uid}__f{i:05d}.jpg'
                Image.fromarray(a).resize((SIZE,SIZE)).save(d, quality=95)
                written.append(str(d))
        i += 1
    cap.release(); return written

t0, failed = time.time(), []
for n, r in enumerate(todo.itertuples(), 1):
    try:
        faces(r.video_path, LOCAL/r.partition/'fake', r.uid)   # local = fast
    except Exception as e:
        failed.append((r.uid, str(e)))
    if n % 100 == 0:
        el = time.time()-t0
        print(f'{n}/{len(todo)} | {el/60:.0f} min | ~{(el/n)*(len(todo)-n)/60:.0f} min left',
              flush=True)
print('extraction done, failures:', len(failed))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
3252 fake videos already extracted
1748 remaining
100/1748 | 8 min | ~137 min left
200/1748 | 18 min | ~142 min left
300/1748 | 28 min | ~134 min left
400/1748 | 38 min | ~128 min left
500/1748 | 47 min | ~118 min left
600/1748 | 57 min | ~110 min left
700/1748 | 67 min | ~100 min left
800/1748 | 76 min | ~90 min left
900/1748 | 85 min | ~80 min left
1000/1748 | 96 min | ~71 min left
1100/1748 | 105 min | ~62 min left
1200/1748 | 116 min | ~53 min left
1300/1748 | 127 min | ~44 min left
1400/1748 | 138 min | ~34 min left
1500/1748 | 149 min | ~25 min left
1600/1748 | 161 min | ~15 min left
1700/1748 | 171 min | ~5 min left
extraction done, failures: 0


Fake frames copied to Drive and the frame index rebuilt. The partition counts printed here are those in Table 1.

In [ ]:
!mkdir -p "{DRIVEF}"/{train,val,test}/fake
!cp -rn /content/ffpp_frames/train/fake/. "{DRIVEF}/train/fake/"
!cp -rn /content/ffpp_frames/val/fake/.   "{DRIVEF}/val/fake/"
!cp -rn /content/ffpp_frames/test/fake/.  "{DRIVEF}/test/fake/"

man = pd.read_csv(OUT/'manifests'/'ffpp_manifest.csv')
man['uid'] = np.where(man.label==1, man.method+'__'+man.video_id, man.video_id)
meta = man.set_index('uid')[['group_id','partition','method','label']]

rows = [dict(frame_path=str(p), uid=p.name.split('__f')[0], partition=part,
             is_fake=lab=='fake')
        for part in ('train','val','test') for lab in ('real','fake')
        for p in (DRIVEF/part/lab).glob('*.jpg')]
idx = pd.DataFrame(rows).join(meta, on='uid', rsuffix='_m')
idx = idx.rename(columns={'uid':'video_id'})[
    ['frame_path','video_id','group_id','label','method','partition']]
idx.to_csv(OUT/'frames'/'ffpp_frames.csv', index=False)

assert idx.frame_path.is_unique and idx.video_id.nunique()==6000, 'index problem'
assert idx.label.notna().all(), 'unmatched frames'
for a in ('train','val','test'):
    for b in ('train','val','test'):
        if a<b: assert not (set(idx[idx.partition==a].group_id) &
                            set(idx[idx.partition==b].group_id)), f'{a}/{b} LEAK'
print(len(idx), 'frames |', idx.video_id.nunique(), 'videos')
print(idx.pivot_table(index='partition', columns='label', values='frame_path',
                      aggfunc='count', fill_value=0))
print('detection rate %.1f%%' % (100*len(idx)/(K*idx.video_id.nunique())))

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
119976 frames | 6000 videos
label          0      1
partition              
test        2799  13997
train      14399  71989
val         2800  13992
detection rate 100.0%


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time, random, json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
from sklearn.metrics import roc_auc_score

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic=True
BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
(OUT/'models').mkdir(exist_ok=True); (OUT/'results').mkdir(exist_ok=True)
SIZE,BATCH,EPOCHS,PATIENCE,LR_HEAD,LR_BODY=300,32,5,3,1e-3,1e-4
CKPT=OUT/'models'/'ffpp_clean_last.pth'; BEST=OUT/'models'/'efficientnetv2s_ffpp_clean.pth'

idx=pd.read_csv(OUT/'frames'/'ffpp_frames.csv')
tr,va=idx[idx.partition=='train'],idx[idx.partition=='val']
assert not set(tr.group_id)&set(va.group_id)
print(f'train {len(tr)} | val {len(va)}',flush=True)

TTF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(.1,.1,.1),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
ETF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],[.229,.224,.225])])

class DS(Dataset):
    def __init__(s,df,tf): s.p=df.frame_path.tolist(); s.y=df.label.tolist(); s.tf=tf
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        for _ in range(3):
            try: return s.tf(Image.open(s.p[i]).convert('RGB')), s.y[i]
            except OSError: time.sleep(1)
        return torch.zeros(3,SIZE,SIZE), s.y[i]

dev='cuda'
m=efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
m.classifier[1]=nn.Linear(m.classifier[1].in_features,2); m=m.to(dev)
head=list(m.classifier.parameters()); hid={id(p) for p in head}
opt=torch.optim.AdamW([{'params':[p for p in m.parameters() if id(p) not in hid],'lr':LR_BODY},
                       {'params':head,'lr':LR_HEAD}],weight_decay=1e-4)
crit=nn.CrossEntropyLoss(); sc=torch.amp.GradScaler('cuda')

start,best,bad,hist=1,-1,0,[]
if CKPT.exists():
    ck=torch.load(CKPT,map_location=dev)
    m.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); sc.load_state_dict(ck['scaler'])
    start,best,bad,hist=ck['epoch']+1,ck['best'],ck['bad'],ck['hist']
    print(f'resumed after epoch {ck["epoch"]}, best val AUC {best:.4f}',flush=True)

g=torch.Generator(); g.manual_seed(SEED)
dtr=DataLoader(DS(tr,TTF),batch_size=BATCH,shuffle=True,num_workers=8,pin_memory=True,
               generator=g,drop_last=True,persistent_workers=True,prefetch_factor=4)
dva=DataLoader(DS(va,ETF),batch_size=BATCH*2,num_workers=8,pin_memory=True)

for ep in range(start,EPOCHS+1):
    m.train(); t0=time.time(); L=[]
    for k,(x,y) in enumerate(dtr,1):
        x,y=x.to(dev,non_blocking=True),y.to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'): loss=crit(m(x),y)
        sc.scale(loss).backward(); sc.step(opt); sc.update(); L.append(loss.item())
        if k%100==0: print(f'  ep{ep} {k}/{len(dtr)} loss {np.mean(L[-100:]):.4f} '
                           f'{(time.time()-t0)/60:.0f}m',flush=True)
    m.eval(); S,Y=[],[]
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for x,y in dva:
            S+=torch.softmax(m(x.to(dev)),1)[:,1].float().cpu().tolist(); Y+=y.tolist()
    auc=roc_auc_score(Y,S)
    hist.append(dict(epoch=ep,loss=float(np.mean(L)),val_auc=float(auc),
                     mins=round((time.time()-t0)/60,1)))
    print(hist[-1],flush=True)
    if auc>best:
        best,bad=auc,0
        torch.save({'model_state':m.state_dict(),'val_auc':auc,'epoch':ep},BEST)
        print('  new best saved',flush=True)
    else:
        bad+=1
    torch.save({'model':m.state_dict(),'opt':opt.state_dict(),'scaler':sc.state_dict(),
                'epoch':ep,'best':best,'bad':bad,'hist':hist},CKPT)
    json.dump({'best_val_auc':best,'history':hist,'batch':BATCH,'seed':SEED,
               'train_frames':len(tr),'val_frames':len(va),
               'gpu':torch.cuda.get_device_name(0)},
              open(OUT/'results'/'ffpp_finetune.json','w'),indent=2)
    if bad>=PATIENCE: print('early stop'); break

print('best val AUC %.4f'%best)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train 86388 | val 16792
  ep1 100/2699 loss 0.4664 5m
  ep1 200/2699 loss 0.3191 10m
  ep1 300/2699 loss 0.2617 16m
  ep1 400/2699 loss 0.2311 21m
  ep1 500/2699 loss 0.2019 26m
  ep1 600/2699 loss 0.1887 31m
  ep1 700/2699 loss 0.1810 36m
  ep1 800/2699 loss 0.1637 42m
  ep1 900/2699 loss 0.1666 46m
  ep1 1000/2699 loss 0.1670 52m
  ep1 1100/2699 loss 0.1407 57m
  ep1 1200/2699 loss 0.1315 62m
  ep1 1300/2699 loss 0.1390 67m
  ep1 1400/2699 loss 0.1249 72m
  ep1 1500/2699 loss 0.1226 77m
  ep1 1600/2699 loss 0.1268 83m
  ep1 1700/2699 loss 0.1138 88m
  ep1 1800/2699 loss 0.1214 93m
  ep1 1900/2699 loss 0.1078 98m
  ep1 2000/2699 loss 0.1157 103m
  ep1 2100/2699 loss 0.1089 108m
  ep1 2200/2699 loss 0.1134 113m
  ep1 2300/2699 loss 0.0973 118m
  ep1 2400/2699 loss 0.0896 124m
  ep1 2500/2699 loss 0.0999 128m
  ep1 2600/2699 loss 0.0949 134m
{'epoch': 1, 'loss

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, torch, torch.nn as nn, time
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

BASE=Path('/content/drive/MyDrive/deepfake_finetuning'); OUT=BASE/'rebuild_2026'
CH=OUT/'features'/'chunks'; CH.mkdir(parents=True,exist_ok=True)
SIZE, CHUNK = 300, 4000
idx=pd.read_csv(OUT/'frames'/'ffpp_frames.csv')

TF=transforms.Compose([transforms.Resize((SIZE,SIZE)),transforms.ToTensor(),
   transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
class DS(Dataset):
    def __init__(s,p): s.p=p
    def __len__(s): return len(s.p)
    def __getitem__(s,i):
        for _ in range(3):
            try: return TF(Image.open(s.p[i]).convert('RGB'))
            except OSError: time.sleep(1)
        return torch.zeros(3,SIZE,SIZE)

def backbone(weights):
    m=efficientnet_v2_s(weights=None if weights else EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    m.classifier[1]=nn.Linear(m.classifier[1].in_features,2)
    if weights: m.load_state_dict(torch.load(weights,map_location='cuda',weights_only=False)['model_state'])
    return m.cuda().eval()

def feats(tag, weights):
    dest=OUT/'features'/f'ffpp_{tag}.npz'
    if dest.exists(): print('cached',dest.name); return np.load(dest,allow_pickle=True)
    m=backbone(weights); store={}
    for part in ('train','val','test'):
        sub=idx[idx.partition==part].reset_index(drop=True)
        n=(len(sub)+CHUNK-1)//CHUNK; parts=[]
        for c in range(n):
            f=CH/f'{tag}_{part}_{c:03d}.npy'
            if f.exists(): parts.append(np.load(f)); continue
            paths=sub.frame_path.iloc[c*CHUNK:(c+1)*CHUNK].tolist()
            for attempt in range(3):
                try:
                    dl=DataLoader(DS(paths),batch_size=64,num_workers=2,
                                  pin_memory=True,timeout=180)   # hang -> error
                    F=[]
                    with torch.no_grad(), torch.amp.autocast('cuda'):
                        for x in dl:
                            F.append(m.avgpool(m.features(x.cuda())).flatten(1).float().cpu().numpy())
                    arr=np.concatenate(F).astype('float32'); np.save(f,arr); parts.append(arr)
                    print(f'  {tag}/{part} chunk {c+1}/{n} ok',flush=True); break
                except RuntimeError as e:
                    print(f'  {tag}/{part} chunk {c+1} attempt {attempt+1} failed: {str(e)[:80]}',flush=True)
            else:
                raise RuntimeError(f'chunk {c} failed 3 times - remount Drive and re-run')
        store[f'X_{part}']=np.concatenate(parts)
        store[f'y_{part}']=sub.label.to_numpy(); store[f'g_{part}']=sub.group_id.to_numpy()
        store[f'v_{part}']=sub.video_id.to_numpy(); store[f'p_{part}']=sub.frame_path.to_numpy()
        assert len(store[f'X_{part}'])==len(sub), f'{part} row mismatch'
        print(part,store[f'X_{part}'].shape,flush=True)
    np.savez_compressed(dest,**store); print('wrote',dest.name)
    return np.load(dest,allow_pickle=True)

d  = feats('finetuned', OUT/'models'/'efficientnetv2s_ffpp_clean.pth')
dz = feats('frozen', None)
print('done')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  finetuned/train chunk 1/22 ok
  finetuned/train chunk 2/22 ok
  finetuned/train chunk 3/22 ok
  finetuned/train chunk 4/22 ok
  finetuned/train chunk 5/22 ok
  finetuned/train chunk 6/22 ok
  finetuned/train chunk 7/22 ok
  finetuned/train chunk 8/22 ok
  finetuned/train chunk 9/22 ok
  finetuned/train chunk 10/22 ok
  finetuned/train chunk 11/22 ok
  finetuned/train chunk 12/22 ok
  finetuned/train chunk 13/22 ok
  finetuned/train chunk 14/22 ok
  finetuned/train chunk 15/22 ok
  finetuned/train chunk 16/22 ok
  finetuned/train chunk 17/22 ok
  finetuned/train chunk 18/22 ok
  finetuned/train chunk 19/22 ok
  finetuned/train chunk 20/22 ok
  finetuned/train chunk 21/22 ok
  finetuned/train chunk 22/22 ok
train (86388, 1280)
  finetuned/val chunk 1/5 ok
  finetuned/val chunk 2/5 ok
  finetuned/val chunk 3/5 ok
  finetuned/val chunk 4/5 ok
  finetuned/val ch

100%|██████████| 82.7M/82.7M [00:00<00:00, 233MB/s]


  frozen/train chunk 1/22 ok
  frozen/train chunk 2/22 ok
  frozen/train chunk 3/22 ok
  frozen/train chunk 4/22 ok
  frozen/train chunk 5/22 ok
  frozen/train chunk 6/22 ok
  frozen/train chunk 7/22 ok
  frozen/train chunk 8/22 ok
  frozen/train chunk 9/22 ok
  frozen/train chunk 10/22 ok
  frozen/train chunk 11/22 ok
  frozen/train chunk 12/22 ok
  frozen/train chunk 13/22 ok
  frozen/train chunk 14/22 ok
  frozen/train chunk 15/22 ok
  frozen/train chunk 16/22 ok
  frozen/train chunk 17/22 ok
  frozen/train chunk 18/22 ok
  frozen/train chunk 19/22 ok
  frozen/train chunk 20/22 ok
  frozen/train chunk 21/22 ok
  frozen/train chunk 22/22 ok
train (86388, 1280)
  frozen/val chunk 1/5 ok
  frozen/val chunk 2/5 ok
  frozen/val chunk 3/5 ok
  frozen/val chunk 4/5 ok
  frozen/val chunk 5/5 ok
val (16792, 1280)
  frozen/test chunk 1/5 ok
  frozen/test chunk 2/5 ok
  frozen/test chunk 3/5 ok
  frozen/test chunk 4/5 ok
  frozen/test chunk 5/5 ok
test (16796, 1280)
wrote ffpp_frozen.npz
done


First hyperparameter selection, by cross-validation on the training partition. This was superseded: every candidate scored above 0.999, so selection moved to the validation partition in the next cell (Section 3.4). Kept for transparency; `results/ffpp_indistribution.json` comes from this cell.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, json, time, joblib
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, accuracy_score

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026')
RES=OUT/'results'; MOD=OUT/'models'; RES.mkdir(exist_ok=True)
d=np.load(OUT/'features'/'ffpp_finetuned.npz',allow_pickle=True)
Xtr,ytr,gtr=d['X_train'],d['y_train'],d['g_train']
Xte,yte,gte,vte=d['X_test'],d['y_test'],d['g_test'],d['v_test']
assert not set(gtr)&set(gte), 'train/test leak'

scaler=StandardScaler().fit(Xtr)                 # training data only
Xtr_s,Xte_s=scaler.transform(Xtr),scaler.transform(Xte)

# grid search on 20% of training VIDEO GROUPS, grouped folds
rng=np.random.default_rng(42); G=np.unique(gtr)
m=np.isin(gtr, rng.choice(G,size=int(len(G)*0.2),replace=False))
print(f'grid search on {m.sum()} frames from {len(np.unique(gtr[m]))} groups',flush=True)
cv=StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
gs=GridSearchCV(SVC(kernel='rbf',cache_size=2000),
                {'C':[0.1,1,10,100],'gamma':['scale','auto',0.001,0.01]},
                scoring='roc_auc',cv=cv,n_jobs=-1,verbose=1)
t0=time.time(); gs.fit(Xtr_s[m],ytr[m],groups=gtr[m])
print(f'best {gs.best_params_}  grouped-CV AUC {gs.best_score_:.4f}  '
      f'({(time.time()-t0)/60:.0f} min)',flush=True)
pd.DataFrame(gs.cv_results_).to_csv(RES/'ffpp_gridsearch.csv',index=False)

# refit on the full training partition
t0=time.time(); svm=SVC(kernel='rbf',cache_size=4000,**gs.best_params_).fit(Xtr_s,ytr)
print(f'refit on {len(ytr)} frames in {(time.time()-t0)/60:.0f} min | '
      f'support vectors {svm.n_support_.tolist()}',flush=True)

# the test partition, scored once
s=svm.decision_function(Xte_s); p=(s>0).astype(int)
tn,fp,fn,tp=confusion_matrix(yte,p).ravel()
vid=pd.DataFrame(dict(v=vte,y=yte,s=s)).groupby('v').agg(y=('y','first'),s=('s','mean'))
r=dict(auc=roc_auc_score(yte,s), accuracy=accuracy_score(yte,p),
       balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2, f1_weighted=f1_score(yte,p,average='weighted'),
       fake_detection_rate=tp/(tp+fn), false_positive_rate=fp/(fp+tn),
       confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]],
       video_level_auc=roc_auc_score(vid.y,vid.s), n_test_frames=int(len(yte)),
       n_test_videos=int(len(vid)), best_params=gs.best_params_, grouped_cv_auc=gs.best_score_,
       gamma_numeric=float(svm._gamma), n_support=svm.n_support_.tolist(),
       gridsearch_frames=int(m.sum()), calibration='none; AUC from decision_function')
for k,v in r.items(): print(f'  {k:22s} {v}')

joblib.dump({'scaler':scaler,'svm':svm},MOD/'svm_ffpp_clean.pkl')
np.savez_compressed(RES/'ffpp_test_scores.npz',y=yte,scores=s,pred=p,groups=gte,videos=vte)
json.dump(r,open(RES/'ffpp_indistribution.json','w'),indent=2,default=float)
print('saved')

Mounted at /content/drive
grid search on 17273 frames from 72 groups
Fitting 5 folds for each of 16 candidates, totalling 80 fits
best {'C': 0.1, 'gamma': 'auto'}  grouped-CV AUC 0.9996  (11 min)
refit on 86388 frames in 2 min | support vectors [621, 694]
  auc                    0.9618254593064307
  accuracy               0.9561205048821148
  balanced_accuracy      0.9225099733641182
  f1_weighted            0.9561860300421215
  fake_detection_rate    0.972922769164821
  false_positive_rate    0.1279028224365845
  confusion_matrix       [[2441, 358], [379, 13618]]
  video_level_auc        0.9852755102040817
  n_test_frames          16796
  n_test_videos          840
  best_params            {'C': 0.1, 'gamma': 'auto'}
  grouped_cv_auc         0.9996480752101636
  gamma_numeric          0.00078125
  n_support              [621, 694]
  gridsearch_frames      17273
  calibration            none; AUC from decision_function
saved


Final SVM. Hyperparameters selected on the validation partition, refitted on the full training partition, test partition evaluated once.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, json, time, joblib, itertools
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, confusion_matrix, f1_score, accuracy_score

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026'); RES=OUT/'results'; MOD=OUT/'models'
d=np.load(OUT/'features'/'ffpp_finetuned.npz',allow_pickle=True)
Xtr,ytr,gtr=d['X_train'],d['y_train'],d['g_train']
Xva,yva,gva=d['X_val'],d['y_val'],d['g_val']
Xte,yte,gte,vte=d['X_test'],d['y_test'],d['g_test'],d['v_test']
for a,b in [(gtr,gva),(gtr,gte),(gva,gte)]: assert not set(a)&set(b)

scaler=StandardScaler().fit(Xtr)
Xtr_s,Xva_s,Xte_s=(scaler.transform(X) for X in (Xtr,Xva,Xte))
rng=np.random.default_rng(42); G=np.unique(gtr)
m=np.isin(gtr,rng.choice(G,size=int(len(G)*0.2),replace=False))

grid=list(itertools.product([0.01,0.1,1,10,100],['scale','auto',1e-4,1e-3,1e-2],[None,'balanced']))
rows=[]
for C,gam,cw in grid:
    t0=time.time()
    clf=SVC(kernel='rbf',C=C,gamma=gam,class_weight=cw,cache_size=2000).fit(Xtr_s[m],ytr[m])
    rows.append(dict(C=C,gamma=gam,class_weight=cw,
                     val_auc=roc_auc_score(yva,clf.decision_function(Xva_s)),
                     secs=round(time.time()-t0)))
    print({k:rows[-1][k] for k in ('C','gamma','class_weight','val_auc')},flush=True)
g=pd.DataFrame(rows).sort_values('val_auc',ascending=False)
g.astype(str).to_csv(RES/'ffpp_gridsearch_validation.csv',index=False)
b=g.iloc[0]; best=dict(C=b.C,gamma=b.gamma,class_weight=b.class_weight)
print('\nselected on validation:',best,'val AUC %.4f'%b.val_auc)

svm=SVC(kernel='rbf',cache_size=4000,**best).fit(Xtr_s,ytr)
s=svm.decision_function(Xte_s); p=(s>0).astype(int)
tn,fp,fn,tp=confusion_matrix(yte,p).ravel()
vid=pd.DataFrame(dict(v=vte,y=yte,s=s)).groupby('v').agg(y=('y','first'),s=('s','mean'))
r=dict(auc=roc_auc_score(yte,s),video_level_auc=roc_auc_score(vid.y,vid.s),
       accuracy=accuracy_score(yte,p),balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2,
       f1_weighted=f1_score(yte,p,average='weighted'),fake_detection_rate=tp/(tp+fn),
       false_positive_rate=fp/(fp+tn),confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]],
       selection='validation partition',best_params={k:str(v) for k,v in best.items()},
       val_auc=float(b.val_auc),gamma_numeric=float(svm._gamma),n_support=svm.n_support_.tolist())
for k,v in r.items(): print(f'  {k:20s} {v}')
joblib.dump({'scaler':scaler,'svm':svm},MOD/'svm_ffpp_clean.pkl')
np.savez_compressed(RES/'ffpp_test_scores.npz',y=yte,scores=s,pred=p,groups=gte,videos=vte)
json.dump(r,open(RES/'ffpp_indistribution_final.json','w'),indent=2,default=str)
print('saved')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{'C': 0.01, 'gamma': 'scale', 'class_weight': None, 'val_auc': np.float64(0.9772620579106429)}
{'C': 0.01, 'gamma': 'scale', 'class_weight': 'balanced', 'val_auc': np.float64(0.9825528363146289)}
{'C': 0.01, 'gamma': 'auto', 'class_weight': None, 'val_auc': np.float64(0.977239519521359)}
{'C': 0.01, 'gamma': 'auto', 'class_weight': 'balanced', 'val_auc': np.float64(0.982545102303357)}
{'C': 0.01, 'gamma': 0.0001, 'class_weight': None, 'val_auc': np.float64(0.9875741750387977)}
{'C': 0.01, 'gamma': 0.0001, 'class_weight': 'balanced', 'val_auc': np.float64(0.988175539083558)}
{'C': 0.01, 'gamma': 0.001, 'class_weight': None, 'val_auc': np.float64(0.9746672843665768)}
{'C': 0.01, 'gamma': 0.001, 'class_weight': 'balanced', 'val_auc': np.float64(0.9810367403822593)}
{'C': 0.01, 'gamma': 0.01, 'class_weight': None, 'val_auc': np.float64(0.9544107091807563)}
{'C': 

Ablation (Table 4): softmax head and linear SVM on the same features. The frozen-backbone condition in this cell was stopped for time and run with a reduced protocol in the next cell.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import numpy as np, pandas as pd, json, itertools, torch
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026'); RES=OUT/'results'; MOD=OUT/'models'
def load(tag):
    z=np.load(OUT/'features'/f'ffpp_{tag}.npz',allow_pickle=True); return {k:z[k] for k in z.files}

def evaluate(name,y,s,thr,v):
    p=(s>thr).astype(int); tn,fp,fn,tp=confusion_matrix(y,p).ravel()
    vid=pd.DataFrame(dict(v=v,y=y,s=s)).groupby('v').agg(y=('y','first'),s=('s','mean'))
    r=dict(condition=name,auc=roc_auc_score(y,s),video_level_auc=roc_auc_score(vid.y,vid.s),
           accuracy=accuracy_score(y,p),balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2,
           fake_detection_rate=tp/(tp+fn),false_positive_rate=fp/(fp+tn),
           confusion_matrix=[[int(tn),int(fp)],[int(fn),int(tp)]])
    print(f"{name:30s} AUC {r['auc']:.4f} | video {r['video_level_auc']:.4f} | "
          f"bal.acc {r['balanced_accuracy']:.4f} | FPR {r['false_positive_rate']:.4f}",flush=True)
    return r

def select_fit_test(name,d,make,grid):              # identical protocol to the headline
    sc=StandardScaler().fit(d['X_train'])
    Xtr,Xva,Xte=(sc.transform(d[k]) for k in ('X_train','X_val','X_test'))
    rng=np.random.default_rng(42); G=np.unique(d['g_train'])
    m=np.isin(d['g_train'],rng.choice(G,size=int(len(G)*0.2),replace=False))
    best,bauc=None,-1
    for prm in grid:
        a=roc_auc_score(d['y_val'],make(**prm).fit(Xtr[m],d['y_train'][m]).decision_function(Xva))
        if a>bauc: best,bauc=prm,a
    clf=make(**best).fit(Xtr,d['y_train'])
    r=evaluate(name,d['y_test'],clf.decision_function(Xte),0,d['v_test'])
    r.update(best_params={k:str(v) for k,v in best.items()},val_auc=bauc); return r

ft=load('finetuned'); rows=[]

# 1. the network's own softmax head - computed from the saved features, no image reads
ck=torch.load(MOD/'efficientnetv2s_ffpp_clean.pth',map_location='cpu',weights_only=False)['model_state']
W,b=ck['classifier.1.weight'].float().numpy(),ck['classifier.1.bias'].float().numpy()
def head(X):
    z=X@W.T+b; e=np.exp(z-z.max(1,keepdims=True)); return e[:,1]/e.sum(1)
print('sanity: softmax val AUC %.4f (fine-tune reported 0.9899)'%roc_auc_score(ft['y_val'],head(ft['X_val'])))
rows.append(evaluate('fine-tuned + softmax head',ft['y_test'],head(ft['X_test']),0.5,ft['v_test']))

# 2. linear SVM, same features
lin=[dict(C=C,class_weight=cw) for C in (0.001,0.01,0.1,1,10) for cw in (None,'balanced')]
rows.append(select_fit_test('fine-tuned + linear SVM',ft,
            lambda **k: LinearSVC(dual=False,max_iter=20000,random_state=42,**k),lin))

# 3. frozen ImageNet backbone + RBF SVM, same grid as the headline (slowest - runs last)
rbf=[dict(C=C,gamma=g,class_weight=cw) for C,g,cw in itertools.product(
     [0.01,0.1,1,10,100],['scale','auto',1e-4,1e-3,1e-2],[None,'balanced'])]
rows.append(select_fit_test('frozen ImageNet + RBF SVM',load('frozen'),
            lambda **k: SVC(kernel='rbf',cache_size=2000,**k),rbf))

# headline row, for the table
h=json.load(open(RES/'ffpp_indistribution_final.json'))
rows.insert(0,dict(condition='fine-tuned + RBF SVM (proposed)',**{k:h[k] for k in
  ('auc','video_level_auc','accuracy','balanced_accuracy','fake_detection_rate','false_positive_rate')}))

t=pd.DataFrame(rows)[['condition','auc','video_level_auc','balanced_accuracy','false_positive_rate']]
print('\n',t.round(4).to_string(index=False))
t.to_csv(RES/'ffpp_ablation.csv',index=False)
json.dump(rows,open(RES/'ffpp_ablation.json','w'),indent=2,default=str)

Mounted at /content/drive
sanity: softmax val AUC 0.9899 (fine-tune reported 0.9899)
fine-tuned + softmax head      AUC 0.9850 | video 0.9922 | bal.acc 0.9273 | FPR 0.1168
fine-tuned + linear SVM        AUC 0.9831 | video 0.9924 | bal.acc 0.9332 | FPR 0.1008


Frozen ImageNet backbone with a reduced grid and 30 % of the training groups (see the note to Table 4).

In [ ]:
import numpy as np, pandas as pd, json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score

OUT=Path('/content/drive/MyDrive/deepfake_finetuning/rebuild_2026'); RES=OUT/'results'
z=np.load(OUT/'features'/'ffpp_frozen.npz',allow_pickle=True); d={k:z[k] for k in z.files}
sc=StandardScaler().fit(d['X_train'])
Xtr,Xva,Xte=(sc.transform(d[k]) for k in ('X_train','X_val','X_test'))
rng=np.random.default_rng(42); G=np.unique(d['g_train'])
sub=np.isin(d['g_train'],rng.choice(G,size=int(len(G)*0.1),replace=False))
fit=np.isin(d['g_train'],rng.choice(G,size=int(len(G)*0.3),replace=False))
print(f'select on {sub.sum()} frames, fit on {fit.sum()}',flush=True)

best,bauc=None,-1
for prm in [dict(C=c,gamma=g,class_weight='balanced') for c in (0.1,1,10) for g in (1e-4,1e-3)]:
    a=roc_auc_score(d['y_val'],SVC(kernel='rbf',cache_size=2000,**prm)
                    .fit(Xtr[sub],d['y_train'][sub]).decision_function(Xva))
    print(prm,'val AUC %.4f'%a,flush=True)
    if a>bauc: best,bauc=prm,a
clf=SVC(kernel='rbf',cache_size=4000,**best).fit(Xtr[fit],d['y_train'][fit])
s=clf.decision_function(Xte); p=(s>0).astype(int)
tn,fp,fn,tp=confusion_matrix(d['y_test'],p).ravel()
vid=pd.DataFrame(dict(v=d['v_test'],y=d['y_test'],s=s)).groupby('v').agg(y=('y','first'),s=('s','mean'))
r=dict(condition='frozen ImageNet + RBF SVM',auc=roc_auc_score(d['y_test'],s),
       video_level_auc=roc_auc_score(vid.y,vid.s),accuracy=accuracy_score(d['y_test'],p),
       balanced_accuracy=(tp/(tp+fn)+tn/(tn+fp))/2,fake_detection_rate=tp/(tp+fn),
       false_positive_rate=fp/(fp+tn),best_params={k:str(v) for k,v in best.items()},
       val_auc=bauc,note='reduced grid; fitted on 30% of training groups (compute)',
       n_support=clf.n_support_.tolist())
print({k:round(v,4) if isinstance(v,float) else v for k,v in r.items()})
json.dump(r,open(RES/'ffpp_frozen_ablation.json','w'),indent=2,default=str)

select on 8638 frames, fit on 25914
{'C': 0.1, 'gamma': 0.0001, 'class_weight': 'balanced'} val AUC 0.5962
{'C': 0.1, 'gamma': 0.001, 'class_weight': 'balanced'} val AUC 0.6104
{'C': 1, 'gamma': 0.0001, 'class_weight': 'balanced'} val AUC 0.6607
{'C': 1, 'gamma': 0.001, 'class_weight': 'balanced'} val AUC 0.6640
{'C': 10, 'gamma': 0.0001, 'class_weight': 'balanced'} val AUC 0.6812
{'C': 10, 'gamma': 0.001, 'class_weight': 'balanced'} val AUC 0.6674
{'condition': 'frozen ImageNet + RBF SVM', 'auc': np.float64(0.7192), 'video_level_auc': np.float64(0.7666), 'accuracy': 0.7426, 'balanced_accuracy': np.float64(0.6518), 'fake_detection_rate': np.float64(0.788), 'false_positive_rate': np.float64(0.4845), 'best_params': {'C': '10', 'gamma': '0.0001', 'class_weight': 'balanced'}, 'val_auc': np.float64(0.6812), 'note': 'reduced grid; fitted on 30% of training groups (compute)', 'n_support': [3042, 13207]}
